Updated preprocessing to encode binary variables for caps on SQFT and NWKER

In [34]:
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler

SEED = 42
cwd = Path.cwd()
root = cwd.parent

raw = pd.read_csv(root / 'cbecs2018_notebook/data/raw/cbecs2018_final_public.csv')
output_dir = cwd
numeric_candidates = ['SQFT', 'FLCEILHT', 'WKHRS', 'NWKER', 'MONUSE', 'HDD65', 'CDD65', 'HEATP', 'COOLP', 'OCCUPYP']
categorical_candidates = ['CENDIV', 'PBA', 'PUBCLIM', 'WLCNS', 'RFCNS', 'RFTILT', 'BLDSHP',
                          'GLSSPC', 'NFLOOR', 'YRCONC', 'ATTIC', 'MAINHT', 'MAINCL']
assert raw.shape == (6436, 1249) and raw.PUBID.is_unique
print(f'Source: {len(raw):,} buildings. Candidate source variables: 23.')

Source: 6,436 buildings. Candidate source variables: 23.


In [ ]:
missing_target = raw.MFBTU.isna()
assert raw.loc[missing_target, 'ZMFBTU'].eq(9).all()
data = raw.loc[~missing_target].sort_values('PUBID').copy()
excluded = raw.loc[missing_target, ['PUBID', 'ZMFBTU']].copy()
excluded['reason'] = 'MFBTU not applicable'
assert data.MFBTU.ge(0).all() and np.isfinite(data.MFBTU).all()
assert data.FINALWT.gt(0).all()
fit_rows, held_out = train_test_split(data.index, test_size=0.30, random_state=SEED)
validation_rows, test_rows = train_test_split(held_out, test_size=0.50, random_state=SEED)
split_rows = {'train': sorted(fit_rows), 'validation': sorted(validation_rows), 'test': sorted(test_rows)}
clean = data[numeric_candidates + categorical_candidates].copy()
clean['FLCEILHT_gt50'] = clean.FLCEILHT.eq(995).astype(float)
clean['FLCEILHT'] = clean.FLCEILHT.replace(995, 50)
# We also need to parameterize when number of workers and square feet exceed 5000 and 1M, respectively, since EIA only provides weighted averages above these thresholds
clean['SQFT_gte1m'] = (clean.SQFT >= 1000000).astype(float)
clean['NWKER_gte5k'] = (clean.NWKER >= 5000).astype(float)
for col in ['MAINHT', 'MAINCL', 'HEATP', 'COOLP', 'OCCUPYP']:
    assert data.loc[data[col].isna(), 'Z' + col].eq(9).all()
for col in categorical_candidates:
    missing = 'not_applicable' if col in ['MAINHT', 'MAINCL'] else 'unknown'
    clean[col] = clean[col].map(lambda value: missing if pd.isna(value) else f'code_{int(value)}')
clean['NFLOOR'] = clean.NFLOOR.replace({'code_994': '10_to_14', 'code_995': '15_or_more'})
clean['PUBCLIM'] = clean.PUBCLIM.replace({'code_7': 'withheld'})
train = data.loc[split_rows['train']]
training_clean = clean.loc[split_rows['train']]
missing_fraction = training_clean[numeric_candidates].isna().mean()
numeric = missing_fraction.index[missing_fraction <= 0.50].tolist()
categorical = categorical_candidates
source_features = numeric + categorical
y_screen = train.MFBTU.to_numpy()
display(pd.DataFrame({'Buildings': {name: len(rows) for name, rows in split_rows.items()}}))
display(missing_fraction[missing_fraction > 0.50].rename('Training missing fraction').to_frame())
print(f'Screening {len(source_features)} candidate source variables; validation/test are not used.')


,Buildings
train,4449
validation,954
test,954


,Training missing fraction
OCCUPYP,0.71724


Screening 22 candidate source variables; validation/test are not used.


In [36]:
selected = ['SQFT', 'PBA', 'NFLOOR', 'NWKER', 'WKHRS', 'RFTILT', 'MAINHT', 'MAINCL', 'GLSSPC', 'MONUSE']
binary_features = ['SQFT_gte1m','NWKER_gte5k']
selected_numeric = [col for col in numeric if col in selected]
selected_categorical = [col for col in categorical if col in selected] 
features = selected_numeric + selected_categorical + binary_features

In [ ]:
# adding extra parameters for when SQFT and WORKERS exceed maximum thresholds
numeric, categorical = selected_numeric, selected_categorical
imputer = SimpleImputer(strategy='median').fit(clean.loc[split_rows['train'], numeric])
filled = pd.DataFrame(imputer.transform(clean[numeric]), index=clean.index, columns=numeric)
scaler = StandardScaler().fit(filled.loc[split_rows['train']])
encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False, dtype=float)
encoder.fit(clean.loc[split_rows['train'], categorical])
encoded_names = [f'{col}__{level}' for col, levels in zip(categorical, encoder.categories_) for level in levels]

def transform_rows(rows, scale=False):
    values = filled.loc[rows].copy()
    if scale:
        values = pd.DataFrame(scaler.transform(values), index=rows, columns=numeric)
    binary_vals = clean.loc[rows, binary_features].copy()
    encoded = pd.DataFrame(encoder.transform(clean.loc[rows, categorical]), index=rows, columns=encoded_names)
    return pd.concat([values, binary_vals, encoded], axis=1)

X_tree = {name: transform_rows(rows) for name, rows in split_rows.items()}
X_scaled = {name: transform_rows(rows, scale=True) for name, rows in split_rows.items()}
y = {name: data.loc[rows, ['MFBTU']].copy() for name, rows in split_rows.items()}
display(pd.DataFrame({name: {'Rows': len(frame), 'Encoded predictors': frame.shape[1]}
                      for name, frame in X_tree.items()}).T)


,Rows,Encoded predictors
train,4449,93
validation,954,93
test,954,93


In [38]:
# Cell 18: Validate the prepared data
training_numeric = clean.loc[split_rows['train'], numeric]
q1, q3 = training_numeric.quantile(0.25), training_numeric.quantile(0.75)
lower, upper = q1 - 1.5 * (q3 - q1), q3 + 1.5 * (q3 - q1)
outlier_checks = pd.DataFrame({'Lower fence': lower, 'Upper fence': upper,
    'Training rows flagged': ((training_numeric < lower) | (training_numeric > upper)).sum()})
all_rows = [row for rows in split_rows.values() for row in rows]
assert len(all_rows) == len(set(all_rows)) == len(data)
assert set(all_rows) == set(data.index)
feature_order = X_tree['train'].columns.tolist()
assert not {'MFBTU', 'PUBID', 'FINALWT', 'ZMFBTU'}.intersection(feature_order)
for name, rows in split_rows.items():
    assert X_tree[name].columns.tolist() == X_scaled[name].columns.tolist() == feature_order
    assert X_tree[name].index.equals(y[name].index)
    assert np.isfinite(X_tree[name].to_numpy()).all() and np.isfinite(X_scaled[name].to_numpy()).all()
    assert np.array_equal(y[name].MFBTU.to_numpy(), raw.loc[rows, 'MFBTU'].to_numpy())
assert np.allclose(X_scaled['train'][numeric].mean(), 0, atol=1e-10)
assert np.allclose(X_scaled['train'][numeric].std(ddof=0), 1, atol=1e-10)
display(outlier_checks)
print('Checks passed. No outlier removal or target transformation applied to delivered data.')


,Lower fence,Upper fence,Training rows flagged
SQFT,-285000.0,507000.0,277
WKHRS,-75.0,245.0,0
NWKER,-207.5,364.5,572
MONUSE,12.0,12.0,552


Checks passed. No outlier removal or target transformation applied to delivered data.


In [ ]:
metadata_columns = ['PUBID', 'FINALWT', 'ZMFBTU']
membership = data[['PUBID']].copy()
membership['split'] = ''
for name, rows in split_rows.items():
    membership.loc[rows, 'split'] = name
    X_tree[name].to_csv(output_dir / f'X_{name}_tree_upd.csv', index=False, float_format='%.12g')
    X_scaled[name].to_csv(output_dir / f'X_{name}_scaled_upd.csv', index=False, float_format='%.12g')
    y[name].to_csv(output_dir / f'y_{name}_upd.csv', index=False)
combined = membership.join(data[['MFBTU']]).join(pd.concat(X_tree.values()).sort_index())
combined.to_csv(output_dir / 'cbecs2018_processed_upd.csv', index=False, float_format='%.12g')